In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_decision_forests as tfdf

print(f"Found TF-DF {tfdf.__version__}")

# 1. 讀取資料 (直接讀取你上傳到目錄的檔案)
train_df = pd.read_csv("train.csv")
serving_df = pd.read_csv("test.csv")

# 2. 資料前處理函數 (處理姓名與票號)
def preprocess(df):
    df = df.copy()

    def normalize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])

    def ticket_number(x):
        return x.split(" ")[-1]

    def ticket_item(x):
        items = x.split(" ")
        if len(items) == 1:
            return "NONE"
        return "_".join(items[0:-1])

    df["Name"] = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"] = df["Ticket"].apply(ticket_item)
    return df

preprocessed_train_df = preprocess(train_df)
preprocessed_serving_df = preprocess(serving_df)

# 定義要放入模型的輸入特徵
input_features = list(preprocessed_train_df.columns)
input_features.remove("Ticket")
input_features.remove("PassengerId")
if "Survived" in input_features:
    input_features.remove("Survived")

# 3. 將 Pandas DataFrame 轉換為 TensorFlow Dataset
def tokenize_names(features, labels=None):
    """將姓名拆分成 tokens，TF-DF 可以原生處理文字 tokens"""
    features["Name"] =  tf.strings.split(features["Name"])
    return features, labels

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_train_df, label="Survived").map(tokenize_names)
serving_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_serving_df).map(tokenize_names)

# 4. 訓練模型 (使用 Ensemble：結合 100 個 GBT 模型)
predictions = None
num_predictions = 0

print("開始訓練 100 個 GBT 模型組合 (約需2~3分鐘，請稍候)...")
for i in range(100):
    if i % 10 == 0:
        print(f"正在訓練第 {i}/100 個模型...")

    model = tfdf.keras.GradientBoostedTreesModel(
        verbose=0, # 隱藏繁雜的日誌
        features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
        exclude_non_specified_features=True,
        random_seed=i,
        honest=True,
    )
    model.fit(train_ds)

    # 預測存活率
    sub_predictions = model.predict(serving_ds, verbose=0)[:,0]

    # 累加每次的預測結果
    if predictions is None:
        predictions = sub_predictions
    else:
        predictions += sub_predictions
    num_predictions += 1

# 5. 平均預測結果並產出 Kaggle 提交檔
predictions /= num_predictions

kaggle_predictions = pd.DataFrame({
    "PassengerId": serving_df["PassengerId"],
    "Survived": (predictions >= 0.5).astype(int)
})

output_path = "submission_tfdf.csv"
kaggle_predictions.to_csv(output_path, index=False)

Found TF-DF 1.12.0
開始訓練 100 個 GBT 模型組合 (約需2~3分鐘，請稍候)...
正在訓練第 0/100 個模型...


正在訓練第 10/100 個模型...
正在訓練第 20/100 個模型...
正在訓練第 30/100 個模型...
正在訓練第 40/100 個模型...
正在訓練第 50/100 個模型...
正在訓練第 60/100 個模型...
正在訓練第 70/100 個模型...
正在訓練第 80/100 個模型...
正在訓練第 90/100 個模型...
